# Named Entity Recognition: LSTM vs BiLSTM vs BiLSTM+CRF vs DistilBERT

End-to-end sequence labeling pipeline on **CoNLL-2003** (PER, ORG, LOC, MISC):

1. Load and explore the data, IOB tagging
2. GloVe embeddings + character-level CNN for OOV words
3. Three models built from scratch (LSTM, BiLSTM, BiLSTM+CRF with a hand-written CRF layer)
4. Fine-tune DistilBERT with HuggingFace `Trainer`
5. seqeval evaluation, color-coded visualization, comparison, Gradio app

**Kaggle setup (before Run All):**
1. Right side panel > **Session options** > **Accelerator**: GPU T4 x2 (or P100).
2. Same panel > **Internet**: **On** (needed to download CoNLL-2003, GloVe and DistilBERT).
3. Press **Run All**; nothing needs to be installed by hand. Outputs (saved model) go to `/kaggle/working`.

## 1. Imports and reproducibility

In [ ]:
import os
import subprocess
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import seqeval
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seqeval"], check=True)

import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt
from datasets import load_dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from IPython.display import HTML, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
print("device:", device, "| work dir:", WORK_DIR)

try:
    torch.optim.Adam([nn.Parameter(torch.zeros(1))])
    print("torch optimizer check: OK")
except Exception as e:
    raise RuntimeError(
        "PyTorch failed to import its compiler module (usually a half-finished import in this session). "
        "Fix: Run > Restart session, then Run All. If it repeats, enable 'Pin to original environment' in Session options."
    ) from e

## 2. Load the CoNLL-2003 dataset

In [ ]:
dataset = None
for repo in ["eriktks/conll2003", "tomaarsen/conll2003", "conll2003"]:
    try:
        dataset = load_dataset(repo)
        print("loaded from", repo)
        break
    except Exception as e:
        print("failed:", repo, type(e).__name__)

label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in id2label.items()}
print(dataset)
print(label_names)

## 3. Explore the data
Sample sentences, entity counts per split, and sentence length distribution.

In [ ]:
for i in range(3):
    ex = dataset["train"][i]
    print(list(zip(ex["tokens"], [id2label[t] for t in ex["ner_tags"]])))
    print()

rows = []
for split in dataset:
    counter = Counter()
    for tags in dataset[split]["ner_tags"]:
        for t in tags:
            name = id2label[t]
            if name.startswith("B-"):
                counter[name[2:]] += 1
    for ent, n in counter.items():
        rows.append({"split": split, "entity": ent, "count": n})

dist = pd.DataFrame(rows).pivot(index="entity", columns="split", values="count")
display(dist)
dist.plot(kind="bar", figsize=(7, 4), title="Entity counts per split")
plt.show()

lengths = [len(t) for t in dataset["train"]["tokens"]]
plt.figure(figsize=(7, 3))
plt.hist(lengths, bins=40)
plt.title("Sentence length (train)")
plt.show()
print("max length:", max(lengths), "| mean length:", round(float(np.mean(lengths)), 2))

## 4. IOB tagging scheme
The dataset is already IOB2 (`B-` starts an entity, `I-` continues it, `O` is outside). Below are helpers that convert tags to entity spans and back, with a round-trip check on the training set.

In [ ]:
def tags_to_spans(tags):
    spans = []
    start, etype = None, None
    for i, tag in enumerate(list(tags) + ["O"]):
        if tag == "O" or tag.startswith("B-") or (tag.startswith("I-") and tag[2:] != etype):
            if etype is not None:
                spans.append((etype, start, i))
                start, etype = None, None
            if tag != "O":
                etype, start = tag[2:], i
    return spans


def spans_to_iob(n, spans):
    tags = ["O"] * n
    for etype, s, e in spans:
        tags[s] = "B-" + etype
        for k in range(s + 1, e):
            tags[k] = "I-" + etype
    return tags


ok = 0
for tags in dataset["train"]["ner_tags"]:
    names = [id2label[t] for t in tags]
    ok += spans_to_iob(len(names), tags_to_spans(names)) == names
print("round-trip OK:", ok, "/", len(dataset["train"]))

ex = dataset["train"][0]
names = [id2label[t] for t in ex["ner_tags"]]
print([(t, ex["tokens"][s:e]) for t, s, e in tags_to_spans(names)])

## 5. Pre-trained GloVe embeddings
GloVe 100d (Wikipedia + Gigaword) downloaded through gensim.

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")
EMB_DIM = 100
print("glove vocab size:", len(glove.key_to_index))

## 6. Build word and character vocabularies, embedding matrix, OOV statistics
Words are lowercased for the GloVe lookup. Casing information and unseen words are handled by the character-level CNN added to the models later.

In [ ]:
PAD, UNK = "<pad>", "<unk>"

word_counter = Counter(w.lower() for s in dataset["train"]["tokens"] for w in s)
word2idx = {PAD: 0, UNK: 1}
for w in word_counter:
    word2idx[w] = len(word2idx)

for split in ["validation", "test"]:
    for s in dataset[split]["tokens"]:
        for w in s:
            lw = w.lower()
            if lw not in word2idx and lw in glove.key_to_index:
                word2idx[lw] = len(word2idx)

emb_matrix = np.random.normal(0, 0.1, (len(word2idx), EMB_DIM)).astype("float32")
emb_matrix[0] = 0
hits = 0
for w, i in word2idx.items():
    if w in glove.key_to_index:
        emb_matrix[i] = glove[w]
        hits += 1
print(f"vocab: {len(word2idx)} | found in GloVe: {hits} ({hits / len(word2idx):.1%})")

char_counter = Counter(c for s in dataset["train"]["tokens"] for w in s for c in w)
char2idx = {PAD: 0, UNK: 1}
for c in char_counter:
    char2idx[c] = len(char2idx)
MAX_WORD_LEN = 20
print("char vocab:", len(char2idx))

train_words = set(word_counter)
test_tokens = [w.lower() for s in dataset["test"]["tokens"] for w in s]
oov_train = np.mean([w not in train_words for w in test_tokens])
oov_all = np.mean([w not in word2idx for w in test_tokens])
print(f"test tokens unseen in train: {oov_train:.1%} | still OOV after GloVe: {oov_all:.1%}")

## 7. Dataset, padding and DataLoaders
Each batch holds word ids, character ids per word, labels (padding = -100) and sentence lengths.

In [ ]:
class NERDataset(Dataset):
    def __init__(self, split):
        self.tokens = dataset[split]["tokens"]
        self.tags = dataset[split]["ner_tags"]

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, i):
        return encode_tokens(self.tokens[i]) + (self.tags[i],)


def encode_tokens(tokens):
    w = [word2idx.get(t.lower(), 1) for t in tokens]
    c = [[char2idx.get(ch, 1) for ch in t[:MAX_WORD_LEN]] for t in tokens]
    return w, c


def collate_fn(batch):
    B = len(batch)
    lengths = torch.tensor([len(b[0]) for b in batch])
    T = int(lengths.max())
    words = torch.zeros(B, T, dtype=torch.long)
    chars = torch.zeros(B, T, MAX_WORD_LEN, dtype=torch.long)
    labels = torch.full((B, T), -100, dtype=torch.long)
    for i, (w, c, y) in enumerate(batch):
        n = len(w)
        words[i, :n] = torch.tensor(w)
        labels[i, :n] = torch.tensor(y)
        for j, ch in enumerate(c):
            chars[i, j, :len(ch)] = torch.tensor(ch)
    return words, chars, labels, lengths


train_loader = DataLoader(NERDataset("train"), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(NERDataset("validation"), batch_size=64, collate_fn=collate_fn)
test_loader = DataLoader(NERDataset("test"), batch_size=64, collate_fn=collate_fn)

w, c, y, l = next(iter(train_loader))
print(w.shape, c.shape, y.shape, l.shape)

## 8. CRF layer from scratch
Learns a transition matrix between tags (plus start/end scores). Training minimizes the negative log-likelihood (gold path score minus log partition function computed with the forward algorithm). Inference uses Viterbi decoding.

In [ ]:
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.start = nn.Parameter(torch.empty(num_tags).uniform_(-0.1, 0.1))
        self.end = nn.Parameter(torch.empty(num_tags).uniform_(-0.1, 0.1))
        self.trans = nn.Parameter(torch.empty(num_tags, num_tags).uniform_(-0.1, 0.1))

    def _gold_score(self, emissions, tags, mask):
        B, T, _ = emissions.shape
        maskf = mask.float()
        score = self.start[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, T):
            step = self.trans[tags[:, t - 1], tags[:, t]] + emissions[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            score = score + step * maskf[:, t]
        last_idx = mask.long().sum(1) - 1
        last_tags = tags.gather(1, last_idx.unsqueeze(1)).squeeze(1)
        return score + self.end[last_tags]

    def _log_partition(self, emissions, mask):
        B, T, N = emissions.shape
        alpha = self.start.unsqueeze(0) + emissions[:, 0]
        for t in range(1, T):
            scores = alpha.unsqueeze(2) + self.trans.unsqueeze(0) + emissions[:, t].unsqueeze(1)
            nxt = torch.logsumexp(scores, dim=1)
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        return torch.logsumexp(alpha + self.end.unsqueeze(0), dim=1)

    def forward(self, emissions, tags, mask):
        nll = self._log_partition(emissions, mask) - self._gold_score(emissions, tags, mask)
        return nll.mean()

    @torch.no_grad()
    def decode(self, emissions, mask):
        B, T, N = emissions.shape
        score = self.start.unsqueeze(0) + emissions[:, 0]
        identity = torch.arange(N, device=emissions.device).unsqueeze(0).expand(B, N)
        history = []
        for t in range(1, T):
            m = mask[:, t].unsqueeze(1)
            cand = score.unsqueeze(2) + self.trans.unsqueeze(0)
            best_score, best_idx = cand.max(dim=1)
            best_score = best_score + emissions[:, t]
            score = torch.where(m, best_score, score)
            history.append(torch.where(m, best_idx, identity))
        score = score + self.end.unsqueeze(0)
        best_last = score.argmax(dim=1)
        path = [best_last]
        for hist in reversed(history):
            best_last = hist.gather(1, best_last.unsqueeze(1)).squeeze(1)
            path.append(best_last)
        path.reverse()
        return torch.stack(path, dim=1)


crf = CRF(5)
em = torch.randn(2, 4, 5)
tg = torch.randint(0, 5, (2, 4))
mk = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]], dtype=torch.bool)
print("sanity NLL:", crf(em, tg, mk).item(), "| decode shape:", crf.decode(em, mk).shape)

## 9. Model architecture (LSTM / BiLSTM / BiLSTM+CRF)
One configurable class: GloVe word embedding + char-CNN embedding, then LSTM (uni or bi-directional), then a linear layer, then either softmax cross-entropy or the CRF.

In [ ]:
class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim=30, n_filters=50, kernel=3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.conv = nn.Conv1d(char_dim, n_filters, kernel, padding=1)
        self.out_dim = n_filters

    def forward(self, chars):
        B, T, C = chars.shape
        x = self.emb(chars.view(B * T, C)).transpose(1, 2)
        x = torch.relu(self.conv(x)).max(dim=2).values
        return x.view(B, T, -1)


class SequenceTagger(nn.Module):
    def __init__(self, emb_matrix, n_chars, n_tags, bidirectional=False, use_crf=False, hidden=200, dropout=0.5):
        super().__init__()
        self.word_emb = nn.Embedding.from_pretrained(torch.tensor(emb_matrix), freeze=False, padding_idx=0)
        self.char_cnn = CharCNN(n_chars)
        in_dim = emb_matrix.shape[1] + self.char_cnn.out_dim
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), n_tags)
        self.use_crf = use_crf
        self.crf = CRF(n_tags) if use_crf else None

    def emissions(self, words, chars, lengths):
        x = torch.cat([self.word_emb(words), self.char_cnn(chars)], dim=-1)
        x = self.drop(x)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=words.size(1))
        return self.fc(self.drop(out))

    def length_mask(self, lengths, T):
        return torch.arange(T, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)

    def loss(self, words, chars, labels, lengths):
        em = self.emissions(words, chars, lengths)
        if self.use_crf:
            mask = self.length_mask(lengths, words.size(1))
            return self.crf(em, labels.clamp(min=0), mask)
        return nn.functional.cross_entropy(em.reshape(-1, em.size(-1)), labels.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def predict(self, words, chars, lengths):
        em = self.emissions(words, chars, lengths)
        if self.use_crf:
            return self.crf.decode(em, self.length_mask(lengths, words.size(1)))
        return em.argmax(dim=-1)


n_params = sum(p.numel() for p in SequenceTagger(emb_matrix, len(char2idx), len(label_names), True, True).parameters())
print("BiLSTM+CRF parameters:", f"{n_params:,}")

## 10. Training and evaluation utilities
Training keeps the checkpoint with the best validation entity-level F1 (seqeval).

In [ ]:
def predict_loader(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for words, chars, labels, lengths in loader:
        preds = model.predict(words.to(device), chars.to(device), lengths.to(device)).cpu()
        for p, l, n in zip(preds, labels, lengths):
            n = int(n)
            y_pred.append([id2label[int(x)] for x in p[:n]])
            y_true.append([id2label[int(x)] for x in l[:n]])
    return y_true, y_pred


def train_model(name, bidirectional, use_crf, epochs=10, lr=1e-3):
    model = SequenceTagger(emb_matrix, len(char2idx), len(label_names), bidirectional, use_crf).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state, history = 0.0, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for words, chars, labels, lengths in train_loader:
            words, chars, labels, lengths = [t.to(device) for t in (words, chars, labels, lengths)]
            optimizer.zero_grad()
            loss = model.loss(words, chars, labels, lengths)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += loss.item()
        yt, yp = predict_loader(model, val_loader)
        f1 = f1_score(yt, yp)
        history.append({"epoch": epoch, "train_loss": total / len(train_loader), "val_f1": f1})
        print(f"[{name}] epoch {epoch:2d} | loss {total / len(train_loader):.4f} | val F1 {f1:.4f}")
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


models, histories = {}, {}

## 11. Train model 1: LSTM

In [ ]:
models["LSTM"], histories["LSTM"] = train_model("LSTM", bidirectional=False, use_crf=False)

## 12. Train model 2: BiLSTM

In [ ]:
models["BiLSTM"], histories["BiLSTM"] = train_model("BiLSTM", bidirectional=True, use_crf=False)

## 13. Train model 3: BiLSTM + CRF

In [ ]:
models["BiLSTM+CRF"], histories["BiLSTM+CRF"] = train_model("BiLSTM+CRF", bidirectional=True, use_crf=True)

## 14. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, h in histories.items():
    axes[0].plot(h["epoch"], h["train_loss"], label=name)
    axes[1].plot(h["epoch"], h["val_f1"], label=name)
axes[0].set_title("Training loss")
axes[1].set_title("Validation F1 (seqeval)")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## 15. Test-set evaluation with seqeval (from-scratch models)
Entity-level precision, recall and F1 for PER, ORG, LOC and MISC.

In [ ]:
reports = {}
for name, m in models.items():
    yt, yp = predict_loader(m, test_loader)
    reports[name] = (yt, yp)
    print("=" * 60)
    print(name)
    print(classification_report(yt, yp, digits=4))

## 16. Transformer: tokenizer and label alignment
DistilBERT splits words into subwords, so word-level labels must be aligned. The label goes on the first subword of each word; other subwords and special tokens get -100 (ignored by the loss).

In [ ]:
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification, TrainingArguments, Trainer, pipeline)

MODEL_CKPT = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)


def tokenize_and_align_labels(batch):
    enc = tokenizer(batch["tokens"], truncation=True, is_split_into_words=True, max_length=256)
    all_labels = []
    for i, labels in enumerate(batch["ner_tags"]):
        prev = None
        aligned = []
        for wid in enc.word_ids(batch_index=i):
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(labels[wid])
            else:
                aligned.append(-100)
            prev = wid
        all_labels.append(aligned)
    enc["labels"] = all_labels
    return enc


demo = dataset["train"][0]
enc = tokenizer(demo["tokens"], is_split_into_words=True)
print(list(zip(tokenizer.convert_ids_to_tokens(enc["input_ids"]), enc.word_ids())))

tokenized = dataset.map(tokenize_and_align_labels, batched=True, remove_columns=dataset["train"].column_names)
print(tokenized)

## 17. Transformer: model, metrics and Trainer setup

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    yt, yp = [], []
    for p, l in zip(preds, labels):
        yt.append([id2label[int(li)] for li in l if li != -100])
        yp.append([id2label[int(pi)] for pi, li in zip(p, l) if li != -100])
    return {
        "precision": precision_score(yt, yp),
        "recall": recall_score(yt, yp),
        "f1": f1_score(yt, yp),
        "accuracy": accuracy_score(yt, yp),
    }


model_tf = AutoModelForTokenClassification.from_pretrained(
    MODEL_CKPT, num_labels=len(label_names), id2label=id2label, label2id=label2id
)

args = TrainingArguments(
    output_dir=os.path.join(WORK_DIR, "ner-distilbert"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model_tf,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

## 18. Fine-tune DistilBERT

In [ ]:
trainer.train()

## 19. Evaluate DistilBERT on the test set

In [ ]:
out = trainer.predict(tokenized["test"])
preds = np.argmax(out.predictions, axis=-1)
yt, yp = [], []
for p, l in zip(preds, out.label_ids):
    yt.append([id2label[int(li)] for li in l if li != -100])
    yp.append([id2label[int(pi)] for pi, li in zip(p, l) if li != -100])

reports["DistilBERT"] = (yt, yp)
print(classification_report(yt, yp, digits=4))

## 20. Compare all four architectures
Overall scores plus per-entity precision / recall / F1.

In [ ]:
ENTITIES = ["PER", "ORG", "LOC", "MISC"]

summary_rows, entity_rows = [], []
for name, (yt, yp) in reports.items():
    summary_rows.append({
        "model": name,
        "precision": precision_score(yt, yp),
        "recall": recall_score(yt, yp),
        "f1": f1_score(yt, yp),
    })
    rep = classification_report(yt, yp, output_dict=True)
    for ent in ENTITIES:
        entity_rows.append({
            "model": name, "entity": ent,
            "precision": rep[ent]["precision"], "recall": rep[ent]["recall"],
            "f1": rep[ent]["f1-score"], "support": rep[ent]["support"],
        })

summary = pd.DataFrame(summary_rows).set_index("model").round(4)
per_entity = pd.DataFrame(entity_rows).round(4)
display(summary)
display(per_entity)

f1_table = per_entity.pivot(index="entity", columns="model", values="f1")[list(reports.keys())]
f1_table.plot(kind="bar", figsize=(8, 4), title="F1 per entity type and model", ylim=(0, 1))
plt.ylabel("F1")
plt.show()

## 21. Where does the CRF help? Boundary analysis
Two diagnostics: (1) **invalid IOB transitions** (an `I-X` tag that does not follow `B-X` or `I-X`), which a softmax can produce because each token is predicted independently; (2) **boundary errors**: predicted entities of the right type that overlap a gold entity but have the wrong start or end.

In [ ]:
def count_invalid_transitions(seqs):
    bad = 0
    for tags in seqs:
        prev = "O"
        for t in tags:
            if t.startswith("I-") and (prev == "O" or prev[2:] != t[2:]):
                bad += 1
            prev = t
    return bad


def count_boundary_errors(y_true, y_pred):
    n = 0
    for gold, pred in zip(y_true, y_pred):
        gold_spans = set(tags_to_spans(gold))
        pred_spans = set(tags_to_spans(pred))
        for ptype, ps, pe in pred_spans - gold_spans:
            if any(ptype == gtype and ps < ge and gs < pe for gtype, gs, ge in gold_spans):
                n += 1
    return n


diag = []
for name, (yt, yp) in reports.items():
    diag.append({
        "model": name,
        "invalid_transitions": count_invalid_transitions(yp),
        "boundary_errors": count_boundary_errors(yt, yp),
        "f1": round(f1_score(yt, yp), 4),
    })
diag = pd.DataFrame(diag).set_index("model")
display(diag)
diag[["invalid_transitions", "boundary_errors"]].plot(kind="bar", figsize=(8, 4), title="Boundary-related errors on the test set")
plt.show()

### Why the CRF improves boundary detection
A softmax head picks the best tag for each token on its own, so the sequence of tags can be inconsistent (for example `O I-ORG I-ORG`, or `B-PER I-LOC`). The CRF scores the whole tag sequence: its transition matrix learns that `I-X` can only follow `B-X` or `I-X`, and that `B-` tags are likely to follow `O`. Viterbi decoding then returns the best globally consistent path. This mostly helps multi-token entities such as organizations and locations (long names, "of", hyphens), where an entity's start and end tokens are the hardest part. Check the table above: the CRF should show fewer invalid transitions and fewer boundary errors than the plain BiLSTM.

A fine-tuned transformer usually beats all three from-scratch models because its contextual representations are pre-trained on huge corpora. Its subword tokenizer also handles OOV words natively.

## 22. Color-coded entity visualization in the notebook
HTML rendering of predicted spans with one color per entity type, shown for the three from-scratch models.

In [ ]:
COLORS = {"PER": "#ffd54f", "ORG": "#81d4fa", "LOC": "#a5d6a7", "MISC": "#f48fb1"}


def legend_html():
    chips = "".join(
        f'<span style="background:{c};padding:2px 8px;border-radius:6px;margin-right:6px;color:#222;font-size:13px">{e}</span>'
        for e, c in COLORS.items()
    )
    return f'<div style="margin:6px 0">{chips}</div>'


def render_html(tokens, tags):
    start_map = {s: (t, e) for t, s, e in tags_to_spans(tags)}
    parts, i = [], 0
    while i < len(tokens):
        if i in start_map:
            t, e = start_map[i]
            text = " ".join(tokens[i:e])
            parts.append(
                f'<mark style="background:{COLORS[t]};color:#222;padding:2px 6px;border-radius:6px">'
                f'{text} <b style="font-size:0.7em">{t}</b></mark>'
            )
            i = e
        else:
            parts.append(tokens[i])
            i += 1
    return '<div style="font-size:16px;line-height:2.2">' + " ".join(parts) + "</div>"


def tag_text_custom(model, text):
    tokens = re.findall(r"\w+(?:[-']\w+)*|[^\w\s]", text)
    w, c = encode_tokens(tokens)
    words, chars, _, lengths = collate_fn([(w, c, [0] * len(w))])
    model.eval()
    pred = model.predict(words.to(device), chars.to(device), lengths.to(device)).cpu()[0]
    return tokens, [id2label[int(x)] for x in pred]


samples = [
    "Barack Obama visited Microsoft headquarters in Seattle last Tuesday.",
    "Mohamed Salah plays for Liverpool and was born in Egypt.",
    "The European Union and Germany signed a deal with Toyota in Tokyo.",
]
display(HTML(legend_html()))
for name in models:
    display(HTML(f"<b>{name}</b>"))
    for s in samples:
        tokens, tags = tag_text_custom(models[name], s)
        display(HTML(render_html(tokens, tags)))

## 23. Save the fine-tuned model

In [ ]:
SAVE_DIR = os.path.join(WORK_DIR, "ner-distilbert-final")
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("saved to", SAVE_DIR)

## 24. Gradio app: real-time color-coded NER
The text box updates the highlighted spans as you type. On Kaggle the app runs inside the notebook session, so `share=True` is required to get a public link you can open (it needs Internet on). The link stays alive while the session runs.

In [ ]:
try:
    import gradio as gr
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr

ner_pipe = pipeline(
    "token-classification",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)


def ner_predict(text):
    if not text or not text.strip():
        return {"text": "", "entities": []}
    ents = ner_pipe(text)
    return {
        "text": text,
        "entities": [
            {"entity": e["entity_group"], "start": int(e["start"]), "end": int(e["end"])}
            for e in ents
        ],
    }


with gr.Blocks(title="NER Highlighter") as app:
    gr.Markdown("# Named Entity Recognition\nType any text. Entities are highlighted live: **PER**, **ORG**, **LOC**, **MISC**.")
    inp = gr.Textbox(lines=5, label="Input text", placeholder="Type or paste text here...")
    out = gr.HighlightedText(label="Detected entities", color_map=COLORS, show_legend=True)
    inp.change(ner_predict, inputs=inp, outputs=out)
    gr.Examples(
        examples=[
            "Barack Obama met Angela Merkel in Berlin to discuss NATO and the European Union.",
            "Mohamed Salah scored twice for Liverpool at Anfield on Saturday.",
            "Google and Microsoft opened new offices in Cairo and Dubai.",
        ],
        inputs=inp,
    )

app.launch(share=True)

## 25. Conclusions
- Bidirectional context is a big step up from a unidirectional LSTM, because an entity's type often depends on words that come after it.
- The CRF adds sequence-level consistency, which reduces invalid IOB transitions and boundary errors for multi-token entities.
- Fine-tuned DistilBERT gives the best scores and handles OOV words through subword tokenization.
- CoNLL-2003 has no DATE label. To also detect dates, train on a dataset with that tag (for example OntoNotes 5) with the same pipeline.